# Lab 08: Prompt Evaluation and Grading (Gemini 3.1)

Systematic evaluation is critical for ensuring AI reliability. In this lab, we will use **Gemini 3.1 Pro** as a "Judge" to evaluate the outputs of other models or prompts based on strict rubrics.

## Objectives
1. Define a **multi-criteria grading rubric** using Pydantic.
2. Implement a **Model-as-a-Judge** pattern to compare two competing answers.
3. Build an **automated evaluation pipeline** for batch testing.

In [ ]:
import os
from typing import Literal

from dotenv import load_dotenv
from google import genai
from google.genai import types
from IPython.display import Markdown, display
from pydantic import BaseModel, Field

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

print("Client initialized.")

## 1. Defining the Grading Rubric

To get consistent grades, we must provide the judge with a structured schema and clear instructions.

In [ ]:
class EvaluationResult(BaseModel):
    accuracy_score: int = Field(
        description="Score from 1 to 5 based on technical correctness"
    )
    tone_alignment: int = Field(
        description="Score from 1 to 5: Does it follow the persona?"
    )
    conciseness_score: int = Field(
        description="Score from 1 to 5: Is there unnecessary fluff?"
    )
    reasoning: str = Field(
        description="Detailed explanation for the given scores"
    )
    passed_threshold: bool

def evaluate_answer(prompt, answer, expected_persona):
    judge_instruction = (
        f"You are an impartial judge. Evaluate the following answer based on the "
        f"user prompt and the required persona: {expected_persona}."
    )

    eval_contents = f"PROMPT: {prompt}\n\nANSWER TO EVALUATE: {answer}"

    response = client.models.generate_content(
        model="gemini-3.1-pro-preview",
        config=types.GenerateContentConfig(
            system_instruction=judge_instruction,
            response_mime_type="application/json",
            response_schema=EvaluationResult,
        ),
        contents=eval_contents
    )
    return EvaluationResult.model_validate_json(response.text)

# Test the judge with a mediocre answer
test_prompt = "Explain what a black hole is to a 5-year old."
mediocre_answer = (
    "A black hole is a region of spacetime where gravity is so strong that nothing, "
    "including light, can escape."
)

result = evaluate_answer(test_prompt, mediocre_answer, "Simple and playful teacher")
print(result.model_dump_json(indent=2))

## 2. Comparing Two Models (A/B Testing)

One of the most powerful patterns is asking a larger model to choose between two responses. This helps in selecting the best model or the best system prompt.

In [ ]:
class ComparisonResult(BaseModel):
    winner: Literal["Answer A", "Answer B", "Tie"]
    explanation: str
    strengths_of_winner: list[str]

def compare_answers(prompt, answer_a, answer_b):
    comparison_prompt = f"""
    User Prompt: {prompt}

    Answer A: {answer_a}

    Answer B: {answer_b}

    Task: Compare these two answers for helpfulness and clarity. Pick a winner.
    """

    response = client.models.generate_content(
        model="gemini-3.1-pro-preview",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=ComparisonResult,
        ),
        contents=comparison_prompt
    )
    return ComparisonResult.model_validate_json(response.text)

ans_a = (
    "To learn Python, start by installing it and using a text editor. "
    "Write 'print(hello world)'."
)
ans_b = (
    "Start your Python journey by downloading the installer from python.org. "
    "Use an IDE like VS Code, and begin with basic syntax like variables and loops "
    "before moving to functions."
)

comparison = compare_answers("How do I start learning Python?", ans_a, ans_b)
display(Markdown(f"### Winner: {comparison.winner}"))
print(f"Reasoning: {comparison.explanation}")

## 3. Automated Batch Evaluation

Let's simulate a small test suite where we check if multiple answers meet our quality threshold.

In [ ]:
test_cases = [
    {
        "q": "What is 2+2?",
        "a": "The result is 4.",
        "persona": "Concise assistant"
    },
    {
        "q": "Who is CEO of Tesla?",
        "a": "I think it's Bill Gates.",
        "persona": "Fact-checker"
    },
    {
        "q": "Capital of France?",
        "a": "Paris is the capital.",
        "persona": "Normal assistant"
    }
]

print("Starting Batch Evaluation...\n")
for i, case in enumerate(test_cases):
    report = evaluate_answer(case['q'], case['a'], case['persona'])
    status = "✅ PASS" if report.passed_threshold else "❌ FAIL"
    print(f"Case {i+1}: {status} (Accuracy: {report.accuracy_score}/5)")
    if not report.passed_threshold:
        print(f"   Reason: {report.reasoning}")

## Summary

In this lab, you built an advanced evaluation framework:
1. **Structured Rubrics**: Used Pydantic to get numeric scores and reasoning.
2. **A/B Comparison**: Used Gemini 3.1 Pro to rank competing answers.
3. **Batch Pipelines**: Automated the checking of multiple responses against quality thresholds.